# Лабораторная работа № 1.5 — QR-алгоритм для собственных значений

Вариант 6. Запускай ячейки сверху вниз. Все исходные данные заданы в ноутбуке, поэтому его можно открыть отдельно от файлов input.txt и output.txt.

Отражения Хаусхолдера строят A=QR. Затем RQ образует новую матрицу с теми же собственными значениями; повторение выделяет диагональные блоки.

## Исходные данные

Числа ниже соответствуют файлу input.txt этой работы.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from typing import List

rows = [[8.0, -1.0, -3.0], [-5.0, 9.0, -8.0], [4.0, -5.0, 7.0], [1e-06]]
A, eps = rows[:-1], rows[-1][0]
print('A =\n', np.array(A), '\nε =', eps)

A =
 [[ 8. -1. -3.]
 [-5.  9. -8.]
 [ 4. -5.  7.]] 
ε = 1e-06


## QR-разложение

Сначала строим ортогональную Q и верхнюю треугольную R, затем повторяем шаг A_(k+1)=R_k Q_k.

In [2]:
def sign(x):
    return -1 if x < 0 else 1


def l2_norm(x):
    n = len(x)
    l2_norm = 0
    for i in range(n):
        l2_norm += x[i] * x[i]
    return math.sqrt(l2_norm)


def matrix_mult(A, B):
    n = len(A)
    return [[sum(A[i][k] * B[k][j] for k in range(n)) for j in range(n)] for i in range(n)]


def transpose(A):
    m = len(A)
    n = len(A[0])
    A_T = [[A[j][i] for j in range(n)] for i in range(m)]
    return A_T


def matrix_subtract(A, B):
    n = len(A)
    return [[A[i][j] - B[i][j] for j in range(len(A[0]))] for i in range(n)]


def householder(A, col):
    n = len(A)
    a = [A[i][col] for i in range(n)]
    v = [0.0] * n

    v[col] = a[col] + sign(a[col]) * l2_norm(a[col:])

    for i in range(col + 1, n):
        v[i] = a[i]

    E = [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]
    squared_norm = sum(value * value for value in v)
    if squared_norm == 0:
        return E
    scalar = 2.0 / squared_norm
    H = matrix_subtract(E, [[scalar * v[i] * v[j] for j in range(n)] for i in range(n)])

    return H


def get_QR(A):
    n = len(A)
    Q = [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]
    R = [row.copy() for row in A]

    for i in range(n-1):
        H = householder(R, i)
        Q = matrix_mult(Q, H)
        R = matrix_mult(H, R)

    return Q, R

In [3]:
Q, R = get_QR(A)
print('||QR-A||max =', np.max(np.abs(np.array(Q)@np.array(R)-A)))
print('||QᵀQ-I||max =', np.max(np.abs(np.array(Q).T@np.array(Q)-np.eye(len(A)))))

||QR-A||max = 3.552713678800501e-15
||QᵀQ-I||max = 3.3306690738754696e-16


## Завершение итераций

Отдельное значение читается из почти изолированного диагонального элемента. Для блока 2×2 решается квадратное характеристическое уравнение.

In [4]:
def solve_quad(a, b, c):
    discr = b * b - 4 * a * c

    if discr < 0:
        real = -b / (2 * a)
        imag = math.sqrt(-discr)/(2 * a)
        return [complex(real, imag), complex(real, -imag)]
        # return [(real + imag * 1j), (real - imag * 1j)]
    else:
        root1 = (-b + math.sqrt(discr)) / (2 * a)
        root2 = (-b - math.sqrt(discr)) / (2 * a)
        return [root1, root2]


def get_roots(A, i):
    n = len(A)
    a11 = A[i][i]
    a12 = A[i][i + 1] if i + 1 < n else 0.0
    a21 = A[i + 1][i] if i + 1 < n else 0.0
    a22 = A[i + 1][i + 1] if i + 1 < n else 0.0

    return solve_quad(1.0, -a11 - a22, a11 * a22 - a12 * a21)


def subdiag_norm(A, i):
    return math.sqrt(sum(A[row][i] ** 2 for row in range(i + 1, len(A))))


def extract_eigenvalues(A, eps):
    """Собственные значения из блоков 1x1 и 2x2."""
    values = []
    n = len(A)
    i = 0
    while i < n:
        if subdiag_norm(A, i) <= eps:
            values.append(A[i][i])
            i += 1
        elif i + 1 < n:
            # Both columns below the 2x2 block must be small. Checking
            # only column i+1 can accept a block still coupled to row i+2.
            coupling = math.sqrt(sum(A[row][col] ** 2
                                     for row in range(i + 2, n)
                                     for col in (i, i + 1)))
            if coupling > eps:
                return None
            values.extend(get_roots(A, i))
            i += 2
        else:
            return None
    return values


def eigenvalue_change(previous, current):
    remaining = list(previous)
    change = 0.0
    for value in current:
        closest = min(range(len(remaining)), key=lambda j: abs(remaining[j] - value))
        change = max(change, abs(remaining.pop(closest) - value))
    return change


def eigenvals_QR(A, eps, max_iter):
    if eps <= 0 or max_iter < 1:
        raise ValueError("eps and max_iter must be positive")
    A_k = [row.copy() for row in A]
    previous = extract_eigenvalues(A_k, eps)
    for iterations in range(1, max_iter + 1):
        Q, R = get_QR(A_k)
        A_k = matrix_mult(R, Q)
        eigenvalues = extract_eigenvalues(A_k, eps)
        if eigenvalues is not None and previous is not None:
            if eigenvalue_change(previous, eigenvalues) <= eps:
                return eigenvalues, iterations, A_k
        previous = eigenvalues
    raise RuntimeError(f"QR iteration did not converge in {max_iter} iterations")

In [5]:
values, iterations, final_A = eigenvals_QR(A, eps, 1000)
print('Собственные значения:', np.round(values, 8))
print('Итераций:', iterations)

Собственные значения: [13.40254105  8.77858759  1.81887136]
Итераций: 12


## Самопроверка

Сравни собственные значения с независимым расчётом NumPy.

In [6]:
reference = np.linalg.eigvals(np.array(A))
assert np.allclose(sorted(values, key=lambda z: z.real), sorted(reference, key=lambda z: z.real), atol=1e-5)
print('NumPy:', np.round(reference, 8))

NumPy: [13.40254105+0.j  8.77858761+0.j  1.81887134+0.j]
